# Phase 3 — Bi-Encoder Architecture & Contrastive Training

We build and train a **bi-encoder** retrieval model on the SQuAD v1.1 pairs from Phase 1.

### Architecture
```
query  → DistilBERT → mean-pool → Linear(768→256) → L2-norm → query_emb
passage → DistilBERT → mean-pool → Linear(768→256) → L2-norm → doc_emb
```
- **Shared encoder** (same weights for query and document)
- **Mean pooling** over all token outputs (more stable than CLS-only)
- **Optional projection** 768 → 256 (faster dot-product search later)
- **L2 normalisation** so cosine similarity == dot product

### Loss: InfoNCE (in-batch negatives)
With batch size B every document in the batch acts as a negative for every other query — giving B−1 free negatives per query.

$$\mathcal{L} = -\frac{1}{B}\sum_{i=1}^{B}\log\frac{\exp(\mathbf{q}_i\cdot\mathbf{d}_i/\tau)}{\sum_{j=1}^{B}\exp(\mathbf{q}_i\cdot\mathbf{d}_j/\tau)}$$

This is equivalent to cross-entropy on the similarity matrix with diagonal targets.

In [1]:
import os, json, time, math, random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm
import matplotlib.pyplot as plt

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

PROCESSED_DIR = Path('../data/processed')
EVAL_DIR      = Path('../evaluation')
MODEL_DIR     = Path('../models')
MODEL_DIR.mkdir(parents=True, exist_ok=True)
EVAL_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f'Device: {DEVICE}')
print(f'PyTorch: {torch.__version__}')

Device: cuda
PyTorch: 2.12.0.dev20260408+cu128


## 1. Load Data

In [2]:
with open(PROCESSED_DIR / 'corpus.json', encoding='utf-8') as f:
    corpus = json.load(f)   # {doc_id: text}

def load_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(l) for l in f]

train_triplets = load_jsonl(PROCESSED_DIR / 'train.jsonl')
val_triplets   = load_jsonl(PROCESSED_DIR / 'val.jsonl')
test_triplets  = load_jsonl(PROCESSED_DIR / 'test.jsonl')

doc_ids   = list(corpus.keys())
doc_texts = [corpus[did] for did in doc_ids]
id_to_idx = {did: i for i, did in enumerate(doc_ids)}

print(f'Corpus:      {len(corpus):,} passages')
print(f'Train:       {len(train_triplets):,} triplets')
print(f'Validation:  {len(val_triplets):,} triplets')
print(f'Test:        {len(test_triplets):,} triplets')

Corpus:      18,891 passages
Train:       70,079 triplets
Validation:  8,759 triplets
Test:        8,761 triplets


## 2. Tokenizer & Dataset

In [3]:
MODEL_NAME = 'distilbert-base-uncased'
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f'Loaded tokenizer: {MODEL_NAME}')

Loaded tokenizer: distilbert-base-uncased


In [4]:
class TripletDataset(Dataset):
    """
    Each item: (query_text, positive_text)
    Negatives are handled implicitly via InfoNCE in-batch negatives.
    """
    def __init__(self, triplets, corpus):
        self.data   = triplets
        self.corpus = corpus

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        t = self.data[idx]
        return t['query'], self.corpus[t['positive_id']]


def collate_fn(batch):
    """Tokenise a batch of (query, positive) string pairs."""
    queries, positives = zip(*batch)
    q_enc = tokenizer(
        list(queries), padding=True, truncation=True,
        max_length=64, return_tensors='pt'
    )
    d_enc = tokenizer(
        list(positives), padding=True, truncation=True,
        max_length=256, return_tensors='pt'
    )
    return q_enc, d_enc


# Config
BATCH_SIZE = 64
NUM_EPOCHS = 5
LR         = 2e-5
TEMPERATURE = 0.07
PROJ_DIM    = 256   # project 768 → 256; set to None to skip

train_ds = TripletDataset(train_triplets, corpus)
val_ds   = TripletDataset(val_triplets,   corpus)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collate_fn, num_workers=0, pin_memory=(DEVICE=='cuda'))
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=collate_fn, num_workers=0)

print(f'Train batches: {len(train_loader):,}  |  Val batches: {len(val_loader):,}')

Train batches: 1,095  |  Val batches: 137


## 3. Bi-Encoder Model

Architecture: `DistilBERT → mean_pool → LayerNorm → Linear(768→256) → L2-norm`

In [5]:
class MeanPooling(nn.Module):
    """Average all non-padding token embeddings."""
    def forward(self, token_embeddings, attention_mask):
        mask = attention_mask.unsqueeze(-1).float()          # (B, T, 1)
        summed = (token_embeddings * mask).sum(dim=1)        # (B, H)
        counts = mask.sum(dim=1).clamp(min=1e-9)             # (B, 1)
        return summed / counts                               # (B, H)


class BiEncoder(nn.Module):
    """
    Shared-weight bi-encoder.
    Both queries and documents pass through the same transformer + pooler.
    """
    def __init__(self, model_name: str, proj_dim: int | None = 256):
        super().__init__()
        self.transformer = AutoModel.from_pretrained(model_name)
        hidden = self.transformer.config.hidden_size   # 768 for distilbert

        self.pooler = MeanPooling()

        if proj_dim is not None:
            self.projection = nn.Sequential(
                nn.LayerNorm(hidden),
                nn.Linear(hidden, proj_dim, bias=False),
            )
            self.emb_dim = proj_dim
        else:
            self.projection = nn.Identity()
            self.emb_dim = hidden

    def encode(self, input_ids, attention_mask):
        out    = self.transformer(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self.pooler(out.last_hidden_state, attention_mask)
        proj   = self.projection(pooled)
        return F.normalize(proj, p=2, dim=-1)   # L2-normalise → cosine == dot

    def forward(self, q_enc, d_enc):
        # Explicitly pass only input_ids + attention_mask so the method works
        # with both BERT (has token_type_ids) and DistilBERT (does not).
        q_emb = self.encode(q_enc['input_ids'], q_enc['attention_mask'])
        d_emb = self.encode(d_enc['input_ids'], d_enc['attention_mask'])
        return q_emb, d_emb


model = BiEncoder(MODEL_NAME, proj_dim=PROJ_DIM).to(DEVICE)
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params:     {total_params:,}')
print(f'Trainable params: {trainable_params:,}')
print(f'Embedding dim:    {model.emb_dim}')

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Total params:     66,561,024
Trainable params: 66,561,024
Embedding dim:    256


## 4. InfoNCE Loss (from scratch)

Given a batch of B (query, positive_doc) pairs:
1. Stack query embeddings → matrix **Q** (B × D)
2. Stack doc embeddings   → matrix **D** (B × D)
3. Similarity matrix **S** = **Q** @ **D**ᵀ / τ  (B × B)
4. Correct pairs are on the diagonal → targets = [0, 1, 2, ..., B−1]
5. Cross-entropy loss on each row of **S**

In [6]:
def info_nce_loss(q_emb: torch.Tensor, d_emb: torch.Tensor, temperature: float = 0.07) -> torch.Tensor:
    """
    Symmetric InfoNCE loss with in-batch negatives.

    Args:
        q_emb: (B, D) L2-normalised query embeddings
        d_emb: (B, D) L2-normalised document embeddings
        temperature: τ — lower values sharpen the distribution

    Returns:
        Scalar loss (mean of query-side and document-side losses)
    """
    # Similarity matrix: S[i, j] = similarity(query_i, doc_j)
    sim = torch.matmul(q_emb, d_emb.T) / temperature   # (B, B)

    # Targets: correct pair for query i is doc i (diagonal)
    targets = torch.arange(q_emb.size(0), device=q_emb.device)

    # Query-to-document: each row is a distribution over docs
    loss_q = F.cross_entropy(sim,   targets)
    # Document-to-query: each column is a distribution over queries (transpose)
    loss_d = F.cross_entropy(sim.T, targets)

    return (loss_q + loss_d) / 2


# Quick sanity check: random embeddings should give ~log(batch_size) loss
_q = F.normalize(torch.randn(8, 256), dim=-1)
_d = F.normalize(torch.randn(8, 256), dim=-1)
_loss = info_nce_loss(_q, _d)
print(f'Random-init loss: {_loss.item():.4f}  (expected ≈ log({8}) = {math.log(8):.4f})')

Random-init loss: 2.2756  (expected ≈ log(8) = 2.0794)


## 5. Training Setup

In [7]:
from transformers import get_linear_schedule_with_warmup

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=0.01,
    eps=1e-8,
)

total_steps  = len(train_loader) * NUM_EPOCHS
warmup_steps = int(0.1 * total_steps)   # 10% warmup

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

print(f'Total training steps: {total_steps:,}')
print(f'Warmup steps:         {warmup_steps:,}')

Total training steps: 5,475
Warmup steps:         547


## 6. Validation — Recall@10 (fast approximate)

We compute Recall@10 on `val_triplets` without building the full corpus index (too slow per epoch).  
Instead we use a **batch-level** evaluation: for each validation batch, compute the in-batch retrieval accuracy.

In [8]:
@torch.no_grad()
def batch_recall_at_k(q_emb, d_emb, k=10):
    """
    In-batch Recall@k: for each query, check if the correct doc
    appears in the top-k of the in-batch similarity scores.
    Correct pair is always the diagonal.
    """
    sim = torch.matmul(q_emb, d_emb.T)          # (B, B)
    topk = sim.topk(min(k, sim.size(1)), dim=1).indices  # (B, k)
    targets = torch.arange(q_emb.size(0), device=q_emb.device).unsqueeze(1)
    hits = (topk == targets).any(dim=1).float()
    return hits.mean().item()


@torch.no_grad()
def validate(model, loader, temperature=TEMPERATURE):
    model.eval()
    total_loss, total_r10, n_batches = 0.0, 0.0, 0
    for q_enc, d_enc in loader:
        q_enc = {k: v.to(DEVICE) for k, v in q_enc.items()}
        d_enc = {k: v.to(DEVICE) for k, v in d_enc.items()}
        q_emb, d_emb = model(q_enc, d_enc)
        loss  = info_nce_loss(q_emb, d_emb, temperature)
        r10   = batch_recall_at_k(q_emb, d_emb, k=10)
        total_loss += loss.item()
        total_r10  += r10
        n_batches  += 1
    model.train()
    return total_loss / n_batches, total_r10 / n_batches


print('Validation helpers ready.')

Validation helpers ready.


## 7. Training Loop

In [ ]:
history = {
    'train_loss': [],
    'val_loss':   [],
    'val_r10':    [],
    'epoch_times': [],
}

best_val_r10  = -1.0
best_epoch    = -1
GRAD_CLIP     = 1.0
LOG_EVERY     = 200   # print running loss every N steps

model.train()

for epoch in range(1, NUM_EPOCHS + 1):
    t_epoch = time.time()
    running_loss = 0.0
    epoch_loss   = 0.0

    for step, (q_enc, d_enc) in enumerate(tqdm(train_loader, desc=f'Epoch {epoch}/{NUM_EPOCHS}'), 1):
        q_enc = {k: v.to(DEVICE) for k, v in q_enc.items()}
        d_enc = {k: v.to(DEVICE) for k, v in d_enc.items()}

        optimizer.zero_grad()
        q_emb, d_emb = model(q_enc, d_enc)
        loss = info_nce_loss(q_emb, d_emb, TEMPERATURE)
        loss.backward()

        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        scheduler.step()

        running_loss += loss.item()
        epoch_loss   += loss.item()

        if step % LOG_EVERY == 0:
            avg = running_loss / LOG_EVERY
            lr  = scheduler.get_last_lr()[0]
            print(f'  step {step:>5} | loss {avg:.4f} | lr {lr:.2e}')
            running_loss = 0.0

    # --- End of epoch ---
    avg_train_loss = epoch_loss / len(train_loader)
    val_loss, val_r10 = validate(model, val_loader)
    elapsed = time.time() - t_epoch

    history['train_loss'].append(avg_train_loss)
    history['val_loss'].append(val_loss)
    history['val_r10'].append(val_r10)
    history['epoch_times'].append(elapsed)

    print(f'\nEpoch {epoch} | train_loss={avg_train_loss:.4f} | '
          f'val_loss={val_loss:.4f} | val_R@10={val_r10:.4f} | '
          f'time={elapsed/60:.1f}m')

    # Save best checkpoint
    if val_r10 > best_val_r10:
        best_val_r10 = val_r10
        best_epoch   = epoch
        torch.save(model.state_dict(), MODEL_DIR / 'best_biencoder.pt')
        print(f'  ✓ New best saved (val R@10={val_r10:.4f})')
    print()

print(f'\nTraining complete. Best val R@10={best_val_r10:.4f} at epoch {best_epoch}')

Epoch 1/5:  18%|█▊        | 200/1095 [01:18<05:49,  2.56it/s]

  step   200 | loss 1.0912 | lr 7.31e-06


## 8. Training Curves

In [ ]:
epochs = list(range(1, len(history['train_loss']) + 1))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Loss curves
axes[0].plot(epochs, history['train_loss'], 'o-', label='Train', color='steelblue')
axes[0].plot(epochs, history['val_loss'],   's-', label='Val',   color='coral')
axes[0].set_title('InfoNCE Loss', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(alpha=0.3)

# Validation Recall@10
axes[1].plot(epochs, history['val_r10'], 'o-', color='seagreen')
axes[1].axhline(0.724, color='coral',    linestyle='--', label='BM25 Recall@10 (baseline)')
axes[1].axhline(0.753, color='steelblue',linestyle='--', label='TF-IDF Recall@10 (baseline)')
axes[1].set_title('Validation Recall@10 (in-batch)', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Recall@10')
axes[1].set_ylim(0, 1.05); axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

# Epoch times
axes[2].bar(epochs, [t/60 for t in history['epoch_times']], color='mediumpurple', edgecolor='white')
axes[2].set_title('Epoch Duration (minutes)', fontweight='bold')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Minutes')
axes[2].grid(alpha=0.3, axis='y')

plt.suptitle('Bi-Encoder Training — DistilBERT + InfoNCE', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(EVAL_DIR / 'training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved training_curves.png')

# Save history as JSON for the report
with open(EVAL_DIR / 'training_history.json', 'w') as f:
    json.dump(history, f, indent=2)
print('Saved training_history.json')

## 9. Load Best Checkpoint & Build Full Corpus Index

We encode every passage in the corpus once and save the resulting embedding matrix.  
At search time we only need to encode the query.

In [ ]:
model.load_state_dict(torch.load(MODEL_DIR / 'best_biencoder.pt', map_location=DEVICE))
model.eval()
print(f'Loaded best checkpoint (epoch {best_epoch}, val R@10={best_val_r10:.4f})')

In [ ]:
@torch.no_grad()
def encode_texts(texts: list[str], batch_size: int = 256, max_length: int = 256) -> np.ndarray:
    """Encode a list of texts into L2-normalised embeddings. Returns (N, D) numpy array."""
    all_embs = []
    for i in tqdm(range(0, len(texts), batch_size), desc='Encoding'):
        batch = texts[i : i + batch_size]
        enc   = tokenizer(
            batch, padding=True, truncation=True,
            max_length=max_length, return_tensors='pt'
        )
        enc   = {k: v.to(DEVICE) for k, v in enc.items()}
        embs  = model.encode(enc['input_ids'], enc['attention_mask'])
        all_embs.append(embs.cpu().numpy())
    return np.vstack(all_embs)


print('Encoding full corpus...')
corpus_embeddings = encode_texts(doc_texts)   # (N_docs, 256)
print(f'Corpus embeddings shape: {corpus_embeddings.shape}')

# Save for Phase 5 (vector search) and Phase 6 (evaluation)
np.save(MODEL_DIR / 'corpus_embeddings.npy', corpus_embeddings)
with open(MODEL_DIR / 'doc_ids.json', 'w') as f:
    json.dump(doc_ids, f)
print('Saved corpus_embeddings.npy and doc_ids.json')

## 10. Full Corpus Evaluation on Test Set

Now we run proper Recall@1/5/10 and MRR on the test set (same setup as Phase 2 baselines).

In [ ]:
@torch.no_grad()
def neural_retrieve(query: str, k: int = 10) -> list:
    """Retrieve top-k doc_ids for a query using the trained bi-encoder."""
    enc    = tokenizer([query], padding=True, truncation=True,
                       max_length=64, return_tensors='pt')
    enc    = {kk: vv.to(DEVICE) for kk, vv in enc.items()}
    q_emb  = model.encode(enc['input_ids'], enc['attention_mask']).cpu().numpy()  # (1, D)
    scores = (q_emb @ corpus_embeddings.T).ravel()     # (N_docs,)
    top_k  = np.argsort(scores)[::-1][:k]
    return [doc_ids[i] for i in top_k]


def evaluate_full(model_name: str, retrieve_fn, triplets: list, ks=(1, 5, 10)) -> dict:
    reciprocal_ranks = []
    hits = {k: 0 for k in ks}
    for t in tqdm(triplets, desc=f'Evaluating {model_name}'):
        ranked     = retrieve_fn(t['query'])
        correct_id = t['positive_id']
        rr = 0.0
        for rank, did in enumerate(ranked[:max(ks)], 1):
            if did == correct_id:
                rr = 1.0 / rank
                break
        reciprocal_ranks.append(rr)
        for k in ks:
            if correct_id in ranked[:k]:
                hits[k] += 1
    n = len(triplets)
    results = {f'Recall@{k}': hits[k] / n for k in ks}
    results['MRR'] = float(np.mean(reciprocal_ranks))
    return results


# Use a subset for speed (1000 test queries is enough for stable estimates)
eval_subset = test_triplets[:1000]

neural_results = evaluate_full('BiEncoder', neural_retrieve, eval_subset)
print('\nBi-Encoder Results:')
for k, v in neural_results.items():
    print(f'  {k}: {v:.4f}')

## 11. Comparison Table

In [ ]:
# Load Phase 2 baseline results
with open(EVAL_DIR / 'baseline_results.json') as f:
    baseline_results = json.load(f)

all_results = {**baseline_results, 'BiEncoder (ours)': neural_results}

metrics = ['Recall@1', 'Recall@5', 'Recall@10', 'MRR']
header  = f"{'Model':<25}" + ''.join(f"{m:<14}" for m in metrics)
print(header)
print('-' * len(header))
for model_name, res in all_results.items():
    row = f"{model_name:<25}" + ''.join(f"{res[m]:<14.4f}" for m in metrics)
    print(row)

# Save combined results
with open(EVAL_DIR / 'all_results.json', 'w') as f:
    json.dump(all_results, f, indent=2)
print('\nSaved all_results.json')

In [ ]:
# Grouped bar chart: all 3 models × 4 metrics
model_names = list(all_results.keys())
x = np.arange(len(metrics))
width = 0.25
colors = ['steelblue', 'coral', 'seagreen']

fig, ax = plt.subplots(figsize=(13, 5))
for i, (mname, color) in enumerate(zip(model_names, colors)):
    vals = [all_results[mname][m] for m in metrics]
    bars = ax.bar(x + i * width - width, vals, width, label=mname,
                  color=color, edgecolor='white', alpha=0.9)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{v:.3f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(metrics, fontsize=12)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score', fontsize=12)
ax.legend(fontsize=11)
ax.set_title('Retrieval Results: TF-IDF vs BM25 vs Bi-Encoder',
             fontsize=14, fontweight='bold')
ax.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(EVAL_DIR / 'full_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved full_comparison.png')

## 12. Qualitative Analysis

Find examples where the bi-encoder succeeds but BM25 fails — the most compelling evidence of semantic understanding.

In [ ]:
# We need BM25 available — reload from Phase 2 artefacts
# (If you are running this notebook fresh, re-run the BM25 build below)
try:
    _ = bm25
    print('BM25 already in memory.')
except NameError:
    from rank_bm25 import BM25Okapi
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity

    print('Rebuilding BM25 and TF-IDF (takes ~30s)...')
    tokenized_corpus = [d.lower().split() for d in tqdm(doc_texts)]
    bm25 = BM25Okapi(tokenized_corpus)

    tfidf = TfidfVectorizer(max_features=50_000, sublinear_tf=True,
                            ngram_range=(1, 2), min_df=2)
    doc_matrix = tfidf.fit_transform(doc_texts)
    print('Done.')


def bm25_retrieve(query, k=10):
    scores = bm25.get_scores(query.lower().split())
    return [doc_ids[i] for i in np.argsort(scores)[::-1][:k]]

def tfidf_retrieve(query, k=10):
    q_vec  = tfidf.transform([query])
    scores = cosine_similarity(q_vec, doc_matrix).ravel()
    return [doc_ids[i] for i in np.argsort(scores)[::-1][:k]]

In [ ]:
print('=== Cases where Bi-Encoder succeeds but BM25 fails ===\n')
shown = 0
for t in test_triplets:
    if shown >= 5:
        break
    q, pos_id = t['query'], t['positive_id']
    neural_top5 = neural_retrieve(q, k=5)
    bm25_top5   = bm25_retrieve(q,   k=5)
    if pos_id in neural_top5 and pos_id not in bm25_top5:
        print(f'Query:    {q}')
        print(f'Passage:  {corpus[pos_id][:250]}...')
        print(f'Neural rank:  {neural_top5.index(pos_id)+1}')
        print(f'BM25 rank:    not in top-5')
        print()
        shown += 1

if shown == 0:
    print('No such examples found in first scan — increase test set search range.')

## 13. Save Model for Phase 5 (Vector Search over Jurafsky & Martin)

In [ ]:
# Save tokenizer alongside model for easy reload
tokenizer.save_pretrained(MODEL_DIR / 'tokenizer')

# Save model config for reconstruction
model_config = {
    'model_name': MODEL_NAME,
    'proj_dim':   PROJ_DIM,
    'emb_dim':    model.emb_dim,
    'temperature': TEMPERATURE,
    'best_epoch':  best_epoch,
    'best_val_r10': best_val_r10,
}
with open(MODEL_DIR / 'model_config.json', 'w') as f:
    json.dump(model_config, f, indent=2)

print('Saved:')
print(f'  {MODEL_DIR}/best_biencoder.pt')
print(f'  {MODEL_DIR}/tokenizer/')
print(f'  {MODEL_DIR}/model_config.json')
print(f'  {MODEL_DIR}/corpus_embeddings.npy')
print(f'  {MODEL_DIR}/doc_ids.json')
print()
print('→ Phase 5 will load best_biencoder.pt + tokenizer to search the Jurafsky & Martin book.')

## Summary

| Component | Choice | Rationale |
|-----------|--------|-----------|
| **Base model** | `distilbert-base-uncased` | 40% smaller than BERT-base, 97% of BERT performance |
| **Pooling** | Mean-pool (all tokens) | More stable than CLS; naturally down-weights padding tokens |
| **Projection** | Linear 768→256 | Reduces search-time compute; slight regularisation effect |
| **Normalisation** | L2-norm | Converts cosine similarity to dot product (cheaper + FAISS-compatible) |
| **Loss** | InfoNCE (symmetric) | B−1 free negatives per query; same as DPR / SimCSE; stable gradient signal |
| **Temperature** | τ = 0.07 | Standard value from MoCo / SimCLR; sharpens similarity distribution |
| **Optimizer** | AdamW, lr=2e-5 | Standard transformer fine-tuning config; weight decay prevents overfitting |
| **Warmup** | 10% of steps | Prevents early instability from large gradient updates |

**This notebook covers Phase 3 (architecture) and Phase 4 (contrastive training) together.**

**Next step → Phase 5: Vector Search over Jurafsky & Martin**